In [1]:
import torch

In [17]:
X = torch.tensor([[1., -1., 7.], [2., 3., 6.]])
X

tensor([[ 1., -1.,  7.],
        [ 2.,  3.,  6.]])

In [18]:
X.exp()

tensor([[2.7183e+00, 3.6788e-01, 1.0966e+03],
        [7.3891e+00, 2.0086e+01, 4.0343e+02]])

In [19]:
X.max(dim=0)

torch.return_types.max(
values=tensor([2., 3., 7.]),
indices=tensor([1, 1, 0]))

In [20]:
X @ X.T

tensor([[51., 41.],
        [41., 49.]])

In [ ]:
X.relu_()

tensor([[1., 0., 7.],
        [2., 3., 6.]])

In [22]:

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else: 
    device = "cpu"
    


In [23]:
device

'cpu'

In [24]:
M = X

In [25]:
M = M.cpu()

In [26]:
M.device

device(type='cpu')

In [27]:
R = M @ M.T
R

tensor([[50., 44.],
        [44., 49.]])

In [28]:
R.device

device(type='cpu')

In [29]:
%timeit M @ M.T

1.94 μs ± 54.2 ns per loop (mean ± std. dev. of 7 runs, 1,000,000 loops each)


In [30]:
M = torch.rand((1000, 1000))
%timeit M @ M.T

7.14 ms ± 213 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [42]:
x = torch.tensor(5., requires_grad=True)
f = x ** 2
f

tensor(25., grad_fn=<PowBackward0>)

In [43]:
f.backward()

In [44]:
x.grad

tensor(10.)

In [45]:
lr = .1
with torch.no_grad():
    x -= lr * x.grad

In [46]:
# alternatively, variable points to same address, so old one gets overridden too
x_detached = x.detach()
x_detached -= lr * x.grad
x

tensor(3., requires_grad=True)

In [ ]:
# Gradienten resetten
x.grad.zero_()

tensor(0.)

In [51]:
# training loop
lr = 0.1
x = torch.tensor(5., requires_grad=True)
for iteration in range(100):
    f = x ** 2
    f.backward()
    with torch.no_grad():
        x -= lr * x.grad
    print(x.grad)
    # important!
    x.grad.zero_()

tensor(10.)
tensor(8.)
tensor(6.4000)
tensor(5.1200)
tensor(4.0960)
tensor(3.2768)
tensor(2.6214)
tensor(2.0972)
tensor(1.6777)
tensor(1.3422)
tensor(1.0737)
tensor(0.8590)
tensor(0.6872)
tensor(0.5498)
tensor(0.4398)
tensor(0.3518)
tensor(0.2815)
tensor(0.2252)
tensor(0.1801)
tensor(0.1441)
tensor(0.1153)
tensor(0.0922)
tensor(0.0738)
tensor(0.0590)
tensor(0.0472)
tensor(0.0378)
tensor(0.0302)
tensor(0.0242)
tensor(0.0193)
tensor(0.0155)
tensor(0.0124)
tensor(0.0099)
tensor(0.0079)
tensor(0.0063)
tensor(0.0051)
tensor(0.0041)
tensor(0.0032)
tensor(0.0026)
tensor(0.0021)
tensor(0.0017)
tensor(0.0013)
tensor(0.0011)
tensor(0.0009)
tensor(0.0007)
tensor(0.0005)
tensor(0.0004)
tensor(0.0003)
tensor(0.0003)
tensor(0.0002)
tensor(0.0002)
tensor(0.0001)
tensor(0.0001)
tensor(9.1344e-05)
tensor(7.3075e-05)
tensor(5.8460e-05)
tensor(4.6768e-05)
tensor(3.7414e-05)
tensor(2.9932e-05)
tensor(2.3945e-05)
tensor(1.9156e-05)
tensor(1.5325e-05)
tensor(1.2260e-05)
tensor(9.8080e-06)
tensor(7.8464e-06)

In [58]:
import sklearn.datasets as ds
import sklearn.model_selection as ms
X, y = ds.fetch_california_housing(return_X_y=True)
X_train, X_test, y_train, y_test = ms.train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = ms.train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [59]:
print(X_train.shape)
print(X_test.shape)
print(X_val.shape)

(13209, 8)
(4128, 8)
(3303, 8)


In [60]:
X_train = torch.FloatTensor(X_train)
X_valid = torch.FloatTensor(X_val)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0, keepdims=True)
stds = X_train.std(dim=0, keepdims=True)
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

In [61]:
y_train = torch.FloatTensor(y_train).reshape(-1, 1)
y_valid = torch.FloatTensor(y_val).reshape(-1, 1)
y_test = torch.FloatTensor(y_test).reshape(-1, 1)

In [77]:
torch.manual_seed(42)
n_features = X_train.shape[1]
w = torch.randn((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

In [79]:
lr = 0.4
n_epochs = 200
for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = ((y_pred - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        b -= lr * b.grad
        w -= lr * w.grad
        b.grad.zero_()
        w.grad.zero_()
    print(loss)
    print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

tensor(0.5129, grad_fn=<MeanBackward0>)
Epoch 1/200, Loss: 0.5129053592681885
tensor(0.5129, grad_fn=<MeanBackward0>)
Epoch 2/200, Loss: 0.5129054188728333
tensor(0.5129, grad_fn=<MeanBackward0>)
Epoch 3/200, Loss: 0.5129053592681885
tensor(0.5129, grad_fn=<MeanBackward0>)
Epoch 4/200, Loss: 0.5129053592681885
tensor(0.5129, grad_fn=<MeanBackward0>)
Epoch 5/200, Loss: 0.5129053592681885
tensor(0.5129, grad_fn=<MeanBackward0>)
Epoch 6/200, Loss: 0.5129053592681885
tensor(0.5129, grad_fn=<MeanBackward0>)
Epoch 7/200, Loss: 0.5129053592681885
tensor(0.5129, grad_fn=<MeanBackward0>)
Epoch 8/200, Loss: 0.5129053592681885
tensor(0.5129, grad_fn=<MeanBackward0>)
Epoch 9/200, Loss: 0.5129054188728333
tensor(0.5129, grad_fn=<MeanBackward0>)
Epoch 10/200, Loss: 0.5129053592681885
tensor(0.5129, grad_fn=<MeanBackward0>)
Epoch 11/200, Loss: 0.5129053592681885
tensor(0.5129, grad_fn=<MeanBackward0>)
Epoch 12/200, Loss: 0.5129053592681885
tensor(0.5129, grad_fn=<MeanBackward0>)
Epoch 13/200, Loss: 0

In [80]:
X_new = X_test[:3]

with torch.no_grad():
    y_pred = X_new @ w + b
y_pred

tensor([[0.7213],
        [1.7607],
        [2.7061]])

In [81]:
y_train[:3]

tensor([[1.7690],
        [1.7330],
        [2.0470]])

In [82]:
import torch.nn as nn
torch.manual_seed(42)

model = nn.Linear(in_features=n_features, out_features=1)

In [83]:
model.bias

Parameter containing:
tensor([0.3117], requires_grad=True)

In [84]:
model.weight

Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076]],
       requires_grad=True)

In [ ]:
model(X_train[:2])

tensor([[0.0786],
        [0.1613]], grad_fn=<AddmmBackward0>)

In [96]:
# safest: clear all forward hooks on this module
model._forward_hooks.clear()

# 1) redefine hook (run this cell)
def test_debug(module, inputs, output):
    print("Hello")
    print("weight:", module.weight.shape)
    print("output:", output.shape)

# 2) register and keep the handle (run this cell once)
handle = model.register_forward_hook(test_debug)

# 3) trigger forward pass
model(X_train[:2])

Hello
weight: torch.Size([1, 8])
output: torch.Size([2, 1])


tensor([[0.0786],
        [0.1613]], grad_fn=<AddmmBackward0>)

In [97]:
model(X_train[:2])

Hello
weight: torch.Size([1, 8])
output: torch.Size([2, 1])


tensor([[0.0786],
        [0.1613]], grad_fn=<AddmmBackward0>)

In [98]:
handle.remove()

In [103]:
optimizer = torch.optim.SGD(model.parameters(), lr=lr)
mse = nn.MSELoss()

In [104]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
    for epoch in range(n_epochs):
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        print(f"Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}")

In [105]:
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/200, Loss: 4.270204544067383
Epoch 2/200, Loss: 0.7620967626571655
Epoch 3/200, Loss: 0.6113183498382568
Epoch 4/200, Loss: 0.5919578075408936
Epoch 5/200, Loss: 0.581246018409729
Epoch 6/200, Loss: 0.5727843046188354
Epoch 7/200, Loss: 0.5656700134277344
Epoch 8/200, Loss: 0.559556782245636
Epoch 9/200, Loss: 0.5542512536048889
Epoch 10/200, Loss: 0.5496227145195007
Epoch 11/200, Loss: 0.5455721616744995
Epoch 12/200, Loss: 0.5420194268226624
Epoch 13/200, Loss: 0.5388972163200378
Epoch 14/200, Loss: 0.5361485481262207
Epoch 15/200, Loss: 0.5337244868278503
Epoch 16/200, Loss: 0.5315828919410706
Epoch 17/200, Loss: 0.5296878814697266
Epoch 18/200, Loss: 0.5280079245567322
Epoch 19/200, Loss: 0.5265161991119385
Epoch 20/200, Loss: 0.5251892805099487
Epoch 21/200, Loss: 0.5240070819854736
Epoch 22/200, Loss: 0.5229517817497253
Epoch 23/200, Loss: 0.5220084190368652
Epoch 24/200, Loss: 0.5211637616157532
Epoch 25/200, Loss: 0.5204060673713684
Epoch 26/200, Loss: 0.519725501537323

In [106]:
X_new = X_test[:3]
with torch.no_grad():
    y_pred = model(X_new)

In [108]:
y_pred

tensor([[0.7213],
        [1.7606],
        [2.7062]])

In [110]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)


In [111]:
lr = .1
optimizer = torch.optim.SGD(model.parameters(), lr=lr)
mse = nn.MSELoss()
train_bgd(model, optimizer, mse, X_train, y_train, n_epochs)

Epoch 1/200, Loss: 4.971917629241943
Epoch 2/200, Loss: 2.0490024089813232
Epoch 3/200, Loss: 1.0055638551712036
Epoch 4/200, Loss: 0.8661571741104126
Epoch 5/200, Loss: 0.7838791608810425
Epoch 6/200, Loss: 0.732201874256134
Epoch 7/200, Loss: 0.6987429261207581
Epoch 8/200, Loss: 0.6764354109764099
Epoch 9/200, Loss: 0.6608064770698547
Epoch 10/200, Loss: 0.6491540670394897
Epoch 11/200, Loss: 0.6398535370826721
Epoch 12/200, Loss: 0.6319313049316406
Epoch 13/200, Loss: 0.6248723864555359
Epoch 14/200, Loss: 0.6183494329452515
Epoch 15/200, Loss: 0.6122236847877502
Epoch 16/200, Loss: 0.6063849329948425
Epoch 17/200, Loss: 0.6007853150367737
Epoch 18/200, Loss: 0.5953795313835144
Epoch 19/200, Loss: 0.5901413559913635
Epoch 20/200, Loss: 0.5850571393966675
Epoch 21/200, Loss: 0.5800724625587463
Epoch 22/200, Loss: 0.575218915939331
Epoch 23/200, Loss: 0.5704955458641052
Epoch 24/200, Loss: 0.5658918619155884
Epoch 25/200, Loss: 0.5614054203033447
Epoch 26/200, Loss: 0.557016432285308

In [112]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)